# RNA-PhysicsNet Training on Colab T4

Three-phase training pipeline:
1. **Phase 1**: Pre-train on PDB RNA CIF structures
2. **Phase 2**: Fine-tune on competition CSV data
3. **Phase 3**: Add MSA features

Target: Colab free tier T4 GPU (15GB VRAM)

In [ ]:
# Cell 1: Install dependencies
!pip install torch onnxruntime numba structlog gemmi tqdm

In [ ]:
# Cell 2: Mount drive / link dataset
import os
import sys

# For Colab: mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running on Colab')

# Add repo to path
sys.path.insert(0, '.')

# Dataset paths (adjust as needed)
DATASET_DIR = '/kaggle/input/stanford-rna-3d-folding'
CIF_DIR = os.path.join(DATASET_DIR, 'PDB_RNA')
TRAIN_CSV = os.path.join(DATASET_DIR, 'train_sequences.csv')
LABELS_CSV = os.path.join(DATASET_DIR, 'train_labels.csv')
VAL_CSV = os.path.join(DATASET_DIR, 'validation_sequences.csv')
VAL_LABELS = os.path.join(DATASET_DIR, 'validation_labels.csv')
MSA_DIR = os.path.join(DATASET_DIR, 'MSA')

print(f'Dataset dir exists: {os.path.isdir(DATASET_DIR)}')

In [ ]:
# Cell 3: Import training modules
from topomatrix_rna.config import PipelineConfig
from topomatrix_rna.train import Trainer

config = PipelineConfig()
trainer = Trainer(config)
print(f'Model parameters: {sum(p.numel() for p in trainer.model.parameters()):,}')
print(f'Device: {trainer.device}')

In [ ]:
# Cell 4: Phase 1 — Pre-train on CIF structures
if os.path.isdir(CIF_DIR):
    print('Starting Phase 1: CIF pre-training...')
    trainer.phase1_pretrain(CIF_DIR, n_epochs=50)
else:
    print(f'CIF directory not found: {CIF_DIR}')
    print('Skipping Phase 1')

In [ ]:
# Cell 5: Phase 2 — Fine-tune on competition data
if os.path.isfile(TRAIN_CSV):
    print('Starting Phase 2: Competition data fine-tuning...')
    trainer.phase2_finetune(
        TRAIN_CSV, LABELS_CSV,
        val_csv=VAL_CSV, val_labels=VAL_LABELS,
        n_epochs=100,
    )
else:
    print(f'Training CSV not found: {TRAIN_CSV}')
    print('Skipping Phase 2')

In [ ]:
# Cell 6: Phase 3 — MSA features
if os.path.isdir(MSA_DIR) and os.path.isfile(TRAIN_CSV):
    print('Starting Phase 3: MSA-augmented training...')
    trainer.phase3_msa(TRAIN_CSV, LABELS_CSV, MSA_DIR, n_epochs=50)
else:
    print(f'MSA directory not found: {MSA_DIR}')
    print('Skipping Phase 3')

In [ ]:
# Cell 7: Export ONNX
onnx_path = trainer.export_onnx('topomatrix_rna/models/rna_physicsnet.onnx')
print(f'ONNX model exported to: {onnx_path}')
print(f'File size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB')

In [ ]:
# Cell 8: Validate on validation set
if os.path.isfile(VAL_CSV) and os.path.isfile(VAL_LABELS):
    from topomatrix_rna.memory_manager import CompetitionDataset
    from torch.utils.data import DataLoader

    val_dataset = CompetitionDataset(VAL_CSV, VAL_LABELS)
    val_loader = DataLoader(val_dataset, batch_size=1, num_workers=0)

    mean_tm = trainer.validate(val_loader)
    print(f'Validation TM-score: {mean_tm:.4f}')
else:
    print('Validation data not available')

In [ ]:
# Cell 9: Copy ONNX to repo / drive
import shutil

if os.path.isfile(onnx_path):
    # Copy to drive if available
    drive_path = '/content/drive/MyDrive/rna_physicsnet.onnx'
    try:
        shutil.copy2(onnx_path, drive_path)
        print(f'ONNX copied to: {drive_path}')
    except Exception as e:
        print(f'Could not copy to drive: {e}')
        print(f'ONNX is at: {onnx_path}')
else:
    print('No ONNX file found')